
# Домашнее задание 6.3. Вращение пятна примеси

## 1. Постановка задачи

Рассматривается двумерное уравнение переноса пассивной примеси:

$$
\frac{\partial u}{\partial t}
+v_x\frac{\partial u}{\partial x}
+v_y\frac{\partial u}{\partial y}=0,
$$

где поле скорости задаётся формулами

$$
v_x=-\omega y,\qquad
v_y=\omega x,\qquad
\omega=2\pi.
$$

Расчётная область:

$$
(x,y)\in[-1,1]^2.
$$

Поле скорости соответствует вращению вокруг начала координат с периодом

$$
T=\frac{2\pi}{\omega}=1.
$$

Начальная концентрация представляет собой гладкое внутри носителя компактное пятно:

$$
u_0(x,y)=
\left[
\max\left(
0,\,
1-\frac{(x-0.4)^2+y^2}{r_0^2}
\right)
\right]^3,
\qquad r_0=0.2.
$$

Точное решение получается обратным поворотом координат:

$$
u(x,y,t)=u_0(
x\cos\omega t+y\sin\omega t,\,
-x\sin\omega t+y\cos\omega t
).
$$

После одного полного оборота точное решение совпадает с начальным:

$$
u(x,y,1)=u_0(x,y).
$$

### Численная схема

Используем явную двумерную upwind-схему первого порядка.

Поскольку поле скорости бездивергентно,

$$
\nabla\cdot\mathbf v=0,
$$

уравнение переноса эквивалентно консервативной форме:

$$
u_t+\nabla\cdot(\mathbf v u)=0.
$$

На равномерной сетке обновляем решение по формуле

$$
u_{i,j}^{n+1}
=
u_{i,j}^n
-\frac{\Delta t}{\Delta x}
(F^x_{i+1/2,j}-F^x_{i-1/2,j})
-\frac{\Delta t}{\Delta y}
(F^y_{i,j+1/2}-F^y_{i,j-1/2}).
$$

Направление upwind-потока определяется знаком соответствующей компоненты скорости. Оба пространственных вклада вычисляются из одного и того же слоя $u^n$.

Для устойчивости используем условие CFL:

$$
\boxed{
\Delta t\max_{i,j}
\left(
\frac{|v_{x,i,j}|}{\Delta x}
+
\frac{|v_{y,i,j}|}{\Delta y}
\right)\le1.
}
$$

Выберем число CFL равным $0.8$.

На входящих участках внешней границы концентрация равна нулю. На выходящих участках используем внутреннее значение концентрации, позволяя примеси покидать область.

### Оценка погрешности

Сравнение с точным решением проводится в дискретной норме $L^1$:

$$
E_1(t)=
\Delta x\Delta y
\sum_{i,j}
|u_{i,j}^{\mathrm{num}}(t)
-u(x_i,y_j,t)|.
$$

Исследуем сетки $50\times50$, $100\times100$ и $200\times200$.

Также будем контролировать максимум концентрации и полное количество примеси:

$$
M(t)=\Delta x\Delta y\sum_{i,j}u_{i,j}(t).
$$


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation
from matplotlib.colors import Normalize
from IPython.display import HTML, display

# Параметры задачи
OMEGA = 2 * np.pi
R0 = 0.2
XC = 0.4
YC = 0.0

# Расчётная область
XMIN, XMAX = -1.0, 1.0
YMIN, YMAX = -1.0, 1.0

# Число CFL
CFL = 0.8

# Исследуемые разрешения
GRID_SIZES = [50, 100, 200]


def create_grid(N):
    dx = (XMAX - XMIN) / N
    dy = (YMAX - YMIN) / N

    x = XMIN + (np.arange(N) + 0.5) * dx
    y = YMIN + (np.arange(N) + 0.5) * dy

    X, Y = np.meshgrid(x, y, indexing="xy")

    # Компоненты скорости
    # vx зависит только от y, vy — только от x
    vx = -OMEGA * y
    vy = OMEGA * x

    return X, Y, vx, vy, dx, dy


print(f"Угловая скорость: {OMEGA:.6f}")
print(f"Период вращения: {2*np.pi/OMEGA:.6f}")
print(f"Число CFL: {CFL}")

for N in GRID_SIZES:
    X, Y, vx, vy, dx, dy = create_grid(N)
    print(f"Сетка {N}x{N}: dx={dx:.5f}, dy={dy:.5f}")


In [ ]:

def initial_profile(X, Y, profile="smooth"):
    r2 = (X - XC)**2 + (Y - YC)**2

    if profile == "smooth":
        return np.maximum(
            0.0, 1.0 - r2 / R0**2
        )**3

    elif profile == "slotted":
        inside = r2 <= R0**2

        slit = (
            (np.abs(X - XC) < 0.04)
            & (Y < 0.1)
        )

        return (inside & ~slit).astype(float)

    else:
        raise ValueError("Неизвестный профиль")


def exact_solution(X, Y, t, profile="smooth"):
    theta = OMEGA * t

    # Обратное вращение координат
    X0 = X * np.cos(theta) + Y * np.sin(theta)
    Y0 = -X * np.sin(theta) + Y * np.cos(theta)

    return initial_profile(X0, Y0, profile)


# Проверим начальный профиль и точное решение
X, Y, _, _, _, _ = create_grid(200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, t in zip(axes, [0.0, 0.25, 1.0]):
    U = exact_solution(X, Y, t)

    ax.imshow(
        U,
        origin="lower",
        extent=[XMIN, XMAX, YMIN, YMAX],
        cmap="viridis",
        vmin=0,
        vmax=1,
        interpolation="nearest"
    )

    ax.set_title(f"Точное решение, t={t}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()


In [ ]:

def upwind_step(U, dt, vx, vy, dx, dy, Fx, Fy):
    """
    Один шаг двумерной консервативной upwind-схемы.

    U: концентрация, shape (Ny, Nx)
    vx: скорость по x, shape (Ny,)
    vy: скорость по y, shape (Nx,)

    Fx, Fy: рабочие массивы потоков.
    """

    # Потоки через вертикальные грани
    vxf = vx[:, None]

    Fx[:, 1:-1] = np.where(
        vxf >= 0,
        vxf * U[:, :-1],
        vxf * U[:, 1:]
    )

    # Левая граница:
    # vx > 0 — вход, концентрация снаружи равна нулю
    Fx[:, 0] = np.where(
        vx >= 0, 0.0, vx * U[:, 0]
    )

    # Правая граница:
    # vx < 0 — вход, концентрация снаружи равна нулю
    Fx[:, -1] = np.where(
        vx >= 0, vx * U[:, -1], 0.0
    )

    # Потоки через горизонтальные грани
    vyf = vy[None, :]

    Fy[1:-1, :] = np.where(
        vyf >= 0,
        vyf * U[:-1, :],
        vyf * U[1:, :]
    )

    # Нижняя граница
    Fy[0, :] = np.where(
        vy >= 0, 0.0, vy * U[0, :]
    )

    # Верхняя граница
    Fy[-1, :] = np.where(
        vy >= 0, vy * U[-1, :], 0.0
    )

    # Суммарный выходящий поток через внешнюю границу
    boundary_flux = (
        dy * (Fx[:, -1].sum() - Fx[:, 0].sum())
        + dx * (Fy[-1, :].sum() - Fy[0, :].sum())
    )

    # Одновременное обновление по x и y
    U_new = (
        U
        - dt / dx * (Fx[:, 1:] - Fx[:, :-1])
        - dt / dy * (Fy[1:, :] - Fy[:-1, :])
    )

    return U_new, boundary_flux


In [ ]:

def simulate(
    N,
    profile="smooth",
    t_end=1.0,
    cfl=CFL,
    observation_times=None,
    snapshot_times=()
):
    X, Y, vx, vy, dx, dy = create_grid(N)

    U = initial_profile(X, Y, profile)
    M0 = U.sum() * dx * dy

    # Максимальная скорость расходования CFL
    cfl_rate = (
        np.max(np.abs(vx)) / dx
        + np.max(np.abs(vy)) / dy
    )

    dt_cfl = cfl / cfl_rate

    if not (0 < cfl <= 1):
        raise ValueError("Необходимо 0 < CFL <= 1")

    if observation_times is None:
        observation_times = np.linspace(0, t_end, 81)

    snapshot_times = np.asarray(snapshot_times, dtype=float)

    # Все моменты, в которые нужно получить результат
    times = np.unique(np.concatenate([
        [0.0],
        np.asarray(observation_times, dtype=float),
        snapshot_times,
        [t_end]
    ]))

    if times[0] < 0 or times[-1] > t_end:
        raise ValueError("Времена наблюдений вне интервала")

    saved_keys = set(np.round(snapshot_times, 12))
    snapshots = {}

    # Рабочие массивы для потоков
    Fx = np.empty((N, N + 1))
    Fy = np.empty((N + 1, N))

    errors = []
    maxima = []
    masses = []
    flux_integrals = []
    balance_errors = []
    minima = []

    t = 0.0
    n_steps = 0

    # Интеграл выходящего потока по времени
    total_outflow = 0.0

    for target in times:

        # Интегрируем до очередного момента наблюдения
        while t < target - 1e-13:
            dt = min(dt_cfl, target - t)

            U, boundary_flux = upwind_step(
                U, dt, vx, vy, dx, dy, Fx, Fy
            )

            total_outflow += dt * boundary_flux
            t += dt
            n_steps += 1

        t = float(target)

        # Точное решение в текущий момент
        U_exact = exact_solution(X, Y, t, profile)

        # Интегральная ошибка
        E1 = dx * dy * np.sum(np.abs(U - U_exact))

        # Количество примеси
        M = dx * dy * U.sum()

        errors.append(E1)
        maxima.append(U.max())
        masses.append(M)
        flux_integrals.append(total_outflow)
        balance_errors.append(M + total_outflow - M0)
        minima.append(U.min())

        # Сохраняем выбранные состояния
        key = round(t, 12)

        if key in saved_keys:
            snapshots[key] = U.copy()

    return {
        "N": N,
        "dx": dx,
        "dy": dy,
        "dt_cfl": dt_cfl,
        "cfl": cfl,
        "n_steps": n_steps,
        "times": times,
        "error": np.array(errors),
        "maximum": np.array(maxima),
        "minimum": np.array(minima),
        "mass": np.array(masses),
        "initial_mass": M0,
        "outflow": np.array(flux_integrals),
        "balance_error": np.array(balance_errors),
        "snapshots": snapshots,
        "final": U.copy()
    }


In [ ]:

# Моменты наблюдения для графиков
OBS_TIMES = np.linspace(0, 1, 81)

# Кадры для будущей анимации
ANIM_TIMES = np.linspace(0, 1, 41)

green_results = {}

for N in GRID_SIZES:
    print(f"Расчёт на сетке {N}x{N}...")

    # Для самой подробной сетки сохраняем кадры анимации
    if N == 200:
        save_times = ANIM_TIMES
    else:
        save_times = [0.25, 1.0]

    result = simulate(
        N=N,
        profile="smooth",
        t_end=1.0,
        cfl=0.8,
        observation_times=OBS_TIMES,
        snapshot_times=save_times
    )

    green_results[N] = result

print("\nЧисленные результаты:")
print(
    f"{'N':>5} {'dt_CFL':>12} {'E1(0.25)':>12} "
    f"{'E1(1)':>12} {'max u(1)':>12} "
    f"{'Потеря массы, %':>17}"
)
print("-" * 78)

for N, result in green_results.items():
    idx_quarter = np.argmin(
        np.abs(result["times"] - 0.25)
    )

    E_quarter = result["error"][idx_quarter]
    E_full = result["error"][-1]

    max_full = result["maximum"][-1]

    mass_loss = 100 * (
        1 - result["mass"][-1] / result["initial_mass"]
    )

    print(
        f"{N:5d} "
        f"{result['dt_cfl']:12.6e} "
        f"{E_quarter:12.6e} "
        f"{E_full:12.6e} "
        f"{max_full:12.6f} "
        f"{mass_loss:17.6f}"
    )

    # Проверка неотрицательности и баланса
    print(
        f"  min(u)={result['minimum'].min():.3e}, "
        f"max|ошибки баланса|="
        f"{np.max(np.abs(result['balance_error'])):.3e}"
    )


In [ ]:

N = 200
result = green_results[N]

X, Y, _, _, dx, dy = create_grid(N)

fig, axes = plt.subplots(
    2, 2,
    figsize=(11, 10),
    constrained_layout=True
)

norm = Normalize(vmin=0, vmax=1)

for row, t in enumerate([0.25, 1.0]):
    U_num = result["snapshots"][round(t, 12)]
    U_exact = exact_solution(X, Y, t)

    for col, (U, label) in enumerate([
        (U_num, "Численное решение"),
        (U_exact, "Точное решение")
    ]):
        ax = axes[row, col]

        im = ax.imshow(
            U,
            origin="lower",
            extent=[XMIN, XMAX, YMIN, YMAX],
            cmap="viridis",
            norm=norm,
            interpolation="nearest"
        )

        ax.set_title(f"{label}, t={t}")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_aspect("equal")

fig.colorbar(
    im,
    ax=axes.ravel().tolist(),
    shrink=0.75,
    label="Концентрация u"
)

fig.suptitle(
    "Вращение гладкого пятна, сетка 200x200",
    fontsize=14
)

plt.show()


In [ ]:

N = 200

X, Y, _, _, dx, dy = create_grid(N)

U_num = green_results[N]["final"]
U_exact = exact_solution(X, Y, t=1.0)

error_field = np.abs(U_num - U_exact)

E1 = dx * dy * error_field.sum()
Einf = error_field.max()

fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    error_field,
    origin="lower",
    extent=[XMIN, XMAX, YMIN, YMAX],
    cmap="magma",
    interpolation="nearest"
)

ax.set_title("Абсолютная ошибка после полного оборота")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal")

fig.colorbar(
    im,
    ax=ax,
    label="|u_num - u_exact|"
)

plt.tight_layout()
plt.show()

print(f"Сетка: {N}x{N}")
print(f"Интегральная ошибка E1 = {E1:.8e}")
print(f"Максимальная точечная ошибка = {Einf:.8e}")
print(f"Максимум точного решения = {U_exact.max():.6f}")
print(f"Максимум численного решения = {U_num.max():.6f}")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for N in GRID_SIZES:
    result = green_results[N]

    t = result["times"]

    axes[0].plot(
        t,
        result["error"],
        linewidth=1.6,
        label=f"{N}x{N}"
    )

    axes[1].plot(
        t,
        result["maximum"],
        linewidth=1.6,
        label=f"{N}x{N}"
    )

# Точный максимум непрерывного профиля равен 1
axes[1].axhline(
    1.0,
    color="black",
    linestyle="--",
    linewidth=1,
    label="Точный максимум"
)

axes[0].set_title("Интегральная ошибка E1(t)")
axes[0].set_xlabel("t")
axes[0].set_ylabel("E1")

axes[1].set_title("Максимальная концентрация")
axes[1].set_xlabel("t")
axes[1].set_ylabel("max u")

for ax in axes:
    ax.axvline(0.25, color="gray", linestyle=":", alpha=0.6)
    ax.axvline(1.0, color="gray", linestyle=":", alpha=0.6)
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()



## 2. Анимация вращения гладкого пятна

Построим анимацию точного и численного решений на сетке $200\times200$.

Используются две синхронизированные панели с общей цветовой шкалой $0\le u\le1$ и одинаковым масштабом осей.

Численное решение получено явной двумерной upwind-схемой, а точное — поворотом начального распределения.

Анимация позволяет наблюдать накопление численной диффузии: по мере вращения пятно становится шире, а его максимальная концентрация уменьшается.

В точном решении форма и максимум концентрации сохраняются.


In [ ]:

N = 200

X, Y, _, _, _, _ = create_grid(N)

frames = np.round(ANIM_TIMES, 12)
snapshots = green_results[N]["snapshots"]

fig, axes = plt.subplots(
    1, 2,
    figsize=(10, 4.5),
    dpi=85,
    constrained_layout=True
)

norm = Normalize(vmin=0, vmax=1)

im_num = axes[0].imshow(
    snapshots[frames[0]],
    origin="lower",
    extent=[XMIN, XMAX, YMIN, YMAX],
    cmap="viridis",
    norm=norm,
    interpolation="nearest"
)

im_exact = axes[1].imshow(
    exact_solution(X, Y, frames[0]),
    origin="lower",
    extent=[XMIN, XMAX, YMIN, YMAX],
    cmap="viridis",
    norm=norm,
    interpolation="nearest"
)

axes[0].set_title("Численное решение")
axes[1].set_title("Точное решение")

for ax in axes:
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")

fig.colorbar(
    im_num,
    ax=axes,
    shrink=0.8,
    label="Концентрация"
)

time_title = fig.suptitle("t = 0.000")


def update_animation(frame):
    t = float(frames[frame])

    im_num.set_data(snapshots[frames[frame]])
    im_exact.set_data(exact_solution(X, Y, t))

    time_title.set_text(f"Вращение пятна, t = {t:.3f}")

    return im_num, im_exact, time_title


anim_smooth = FuncAnimation(
    fig,
    update_animation,
    frames=len(frames),
    interval=80,
    blit=False,
    repeat=True
)

plt.close(fig)

# Лимит для встраивания HTML-анимации
plt.rcParams["animation.embed_limit"] = 50

display(
    HTML(anim_smooth.to_jshtml(fps=12))
)



## 3. Выводы по зелёному уровню

Для уравнения переноса пассивной примеси в поле вращения реализована явная двумерная upwind-схема первого порядка с одновременным обновлением по двум пространственным направлениям.

Расчёты выполнены на сетках $50\times50$, $100\times100$ и $200\times200$ при максимальном числе CFL, равном $0.8$.

Построены численное и точное решения через четверть и полный оборот, поле абсолютной ошибки, графики интегральной ошибки и максимальной концентрации, а также анимация вращения пятна.

### Наблюдаемые особенности

При повороте на четверть оборота пятно перемещается из начального положения $(0.4,0)$ в окрестность точки $(0,0.4)$.

После полного оборота точное решение возвращается к начальному распределению.

Численное решение также воспроизводит вращение пятна, но постепенно теряет исходную форму из-за численной диффузии.

При увеличении разрешения сетки интегральная ошибка уменьшается, а максимальная концентрация сохраняется лучше.

### Насколько точно пятно возвращается к исходной форме?

Точное решение возвращается к исходному профилю после одного оборота без изменения формы и амплитуды.

Для upwind-схемы первого порядка возвращение не является точным: численная диффузия приводит к размыванию границ пятна, уменьшению максимума концентрации и увеличению интегральной ошибки.

На грубой сетке эти эффекты выражены сильнее. При измельчении сетки численное решение приближается к точному.

Таким образом, схема правильно воспроизводит направление и период вращения, но сохранение формы ограничено её первым порядком пространственной точности.

### Количество примеси

Для точного решения количество примеси постоянно, поскольку носитель пятна целиком находится внутри области и при вращении не достигает внешней границы.

Для гладкого начального профиля точное количество примеси равно

$$
M(0)=\frac{\pi r_0^2}{4}=0.01\pi.
$$

Численное размытие может привести к появлению небольшого количества примеси у внешней границы и её последующему выходу из области.

При использовании консервативной схемы изменение дискретного количества примеси соответствует накопленному потоку через границу.



## 4. Жёлтый уровень. Вращение диска со щелью

Рассмотрим начальное распределение в форме диска единичной концентрации с центром $(0.4,0)$ и радиусом $r_0=0.2$.

Внутри диска задана узкая щель, в которой концентрация равна нулю:

$$
u_0(x,y)=
\begin{cases}
1, & (x-0.4)^2+y^2\le r_0^2
\text{ и точка вне щели},\\
0, & \text{иначе}.
\end{cases}
$$

Щель определяется условиями

$$
|x-0.4|<0.04,\qquad y<0.1.
$$

Поле скорости и расчётная область не изменяются.

В отличие от гладкого пятна, новый профиль имеет разрывные границы. Поэтому численная диффузия должна особенно сильно влиять на сохранение узкой щели и окружности диска.

### План исследования

1. Построить исходный диск со щелью.
2. Рассчитать вращение на сетках $50\times50$, $100\times100$ и $200\times200$.
3. Сравнить распределения после одного и пяти оборотов.
4. Построить анимацию с наложением контура точного решения.
5. Сравнить максимумы концентрации, ошибки и полное количество примеси для гладкого и разрывного профилей.
6. Проверить дискретный баланс примеси с учётом потока через внешнюю границу.

Поскольку схема консервативна, должно выполняться соотношение

$$
M(t)+\int_0^t\Phi_{\mathrm{out}}(s)\,ds=M(0),
$$

где $\Phi_{\mathrm{out}}$ — суммарный выходящий поток примеси через границу области.

В точном решении примесь остаётся внутри области, но численная диффузия может привести к её распространению до внешней границы.


In [ ]:

N = 200
X, Y, _, _, dx, dy = create_grid(N)

U_smooth = initial_profile(X, Y, "smooth")
U_slotted = initial_profile(X, Y, "slotted")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for ax, U, title in zip(
    axes,
    [U_smooth, U_slotted],
    ["Гладкое пятно", "Диск со щелью"]
):
    im = ax.imshow(
        U,
        origin="lower",
        extent=[XMIN, XMAX, YMIN, YMAX],
        cmap="viridis",
        vmin=0,
        vmax=1,
        interpolation="nearest"
    )

    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")

    # Увеличиваем область вокруг пятна
    ax.set_xlim(0.1, 0.7)
    ax.set_ylim(-0.3, 0.3)

fig.colorbar(
    im,
    ax=axes,
    shrink=0.8,
    label="Концентрация"
)

plt.show()

print("Начальная масса гладкого пятна:",
      dx * dy * U_smooth.sum())

print("Начальная масса диска со щелью:",
      dx * dy * U_slotted.sum())

print("Максимумы концентрации:",
      U_smooth.max(), U_slotted.max())


In [ ]:

# Моменты для исследования динамики
YELLOW_TIMES = np.linspace(0, 5, 101)

# Состояния, которые нужно сохранить
YELLOW_SNAPSHOTS = [0.0, 1.0, 5.0]

yellow_results = {}

for profile in ["smooth", "slotted"]:
    yellow_results[profile] = {}

    for N in GRID_SIZES:
        print(f"Расчёт: {profile}, сетка {N}x{N}")

        result = simulate(
            N=N,
            profile=profile,
            t_end=5.0,
            cfl=0.8,
            observation_times=YELLOW_TIMES,
            snapshot_times=YELLOW_SNAPSHOTS
        )

        yellow_results[profile][N] = result

        print(
            f"  Шаг CFL: {result['dt_cfl']:.6e}, "
            f"шагов: {result['n_steps']}, "
            f"E1(5): {result['error'][-1]:.6e}"
        )

print("\nВсе расчёты завершены.")


In [ ]:

# Подробная сетка только для точного контура
Xc, Yc, _, _, _, _ = create_grid(500)

fig, axes = plt.subplots(
    3, 2,
    figsize=(11, 13),
    constrained_layout=True
)

for i, N in enumerate(GRID_SIZES):
    result = yellow_results["slotted"][N]

    for j, t in enumerate([1.0, 5.0]):
        ax = axes[i, j]

        U = result["snapshots"][round(t, 12)]
        U_exact_fine = exact_solution(
            Xc, Yc, t, "slotted"
        )

        im = ax.imshow(
            U,
            origin="lower",
            extent=[XMIN, XMAX, YMIN, YMAX],
            cmap="viridis",
            vmin=0,
            vmax=1,
            interpolation="nearest"
        )

        # Контур точного диска со щелью
        ax.contour(
            Xc,
            Yc,
            U_exact_fine,
            levels=[0.5],
            colors="red",
            linewidths=1.3
        )

        ax.set_title(
            f"{N}x{N}, {int(t)} оборот(ов)"
        )

        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_aspect("equal")

        # Увеличение области вокруг пятна
        ax.set_xlim(0.1, 0.7)
        ax.set_ylim(-0.3, 0.3)

fig.colorbar(
    im,
    ax=axes.ravel().tolist(),
    shrink=0.65,
    label="Концентрация"
)

fig.suptitle(
    "Сохранение диска со щелью\n"
    "Красный контур — точное решение",
    fontsize=14
)

plt.show()


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

styles = {
    "smooth": "-",
    "slotted": "--"
}

colors_grid = {
    50: "tab:blue",
    100: "tab:orange",
    200: "tab:green"
}

for profile in ["smooth", "slotted"]:
    for N in GRID_SIZES:
        result = yellow_results[profile][N]

        t = result["times"]
        M0 = result["initial_mass"]

        label = f"{profile}, {N}x{N}"

        style = styles[profile]
        color = colors_grid[N]

        # Интегральная ошибка
        axes[0].plot(
            t,
            result["error"],
            linestyle=style,
            color=color,
            linewidth=1.5,
            label=label
        )

        # Максимум концентрации
        axes[1].plot(
            t,
            result["maximum"],
            linestyle=style,
            color=color,
            linewidth=1.5,
            label=label
        )

        # Доля оставшейся примеси
        axes[2].plot(
            t,
            result["mass"] / M0,
            linestyle=style,
            color=color,
            linewidth=1.5,
            label=label
        )

axes[1].axhline(
    1.0,
    color="black",
    linestyle=":",
    label="Точный максимум"
)

axes[2].axhline(
    1.0,
    color="black",
    linestyle=":",
    label="Сохранение массы"
)

titles = [
    "Интегральная ошибка E1(t)",
    "Максимальная концентрация",
    "Доля оставшейся примеси M(t)/M(0)"
]

for ax, title in zip(axes, titles):
    ax.set_title(title)
    ax.set_xlabel("t")
    ax.grid(alpha=0.3)

    for turn in range(1, 6):
        ax.axvline(
            turn,
            color="gray",
            linestyle=":",
            alpha=0.3
        )

axes[0].set_ylabel("E1")
axes[1].set_ylabel("max u")
axes[2].set_ylabel("M(t) / M(0)")

# Общая легенда
handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=3,
    fontsize=9
)

fig.suptitle(
    "Сравнение гладкого пятна и диска со щелью",
    fontsize=14
)

plt.tight_layout(rect=(0, 0.12, 1, 0.95))
plt.show()


In [ ]:

print(
    f"{'Профиль':<10} {'N':>5} {'t':>5} "
    f"{'E1':>13} {'max u':>12} "
    f"{'M/M0':>12} {'Потеря, %':>13}"
)
print("-" * 78)

for profile in ["smooth", "slotted"]:
    for N in GRID_SIZES:
        result = yellow_results[profile][N]

        for time in [1.0, 5.0]:
            idx = np.argmin(
                np.abs(result["times"] - time)
            )

            E1 = result["error"][idx]
            max_u = result["maximum"][idx]

            M = result["mass"][idx]
            M0 = result["initial_mass"]

            relative_mass = M / M0
            loss = 100 * (1 - relative_mass)

            print(
                f"{profile:<10} {N:5d} {time:5.1f} "
                f"{E1:13.6e} {max_u:12.6f} "
                f"{relative_mass:12.8f} "
                f"{loss:13.6f}"
            )


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for N in GRID_SIZES:
    result = yellow_results["slotted"][N]

    t = result["times"]
    M0 = result["initial_mass"]

    # Масса, которая осталась в области
    relative_mass = result["mass"] / M0

    # Накопленная масса, вышедшая через границу
    relative_outflow = result["outflow"] / M0

    # Проверка баланса
    balance = (
        result["mass"] + result["outflow"]
    ) / M0

    axes[0].plot(
        t,
        relative_mass,
        label=f"M(t)/M0, {N}x{N}"
    )

    axes[0].plot(
        t,
        relative_outflow,
        "--",
        label=f"Выходящий поток, {N}x{N}"
    )

    axes[1].plot(
        t,
        balance - 1,
        label=f"{N}x{N}"
    )

axes[0].set_title("Количество примеси и граничный поток")
axes[0].set_xlabel("t")
axes[0].set_ylabel("Доля от начальной массы")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].set_title("Невязка дискретного баланса")
axes[1].set_xlabel("t")
axes[1].set_ylabel("(M + Q_out)/M0 - 1")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Проверка баланса при t=5:\n")

for profile in ["smooth", "slotted"]:
    print(f"Профиль: {profile}")

    for N in GRID_SIZES:
        result = yellow_results[profile][N]

        M0 = result["initial_mass"]
        M = result["mass"][-1]
        Q = result["outflow"][-1]

        balance_error = M + Q - M0

        print(
            f"  {N}x{N}: "
            f"M={M:.10e}, "
            f"Q_out={Q:.10e}, "
            f"невязка={balance_error:.3e}"
        )

    print()



## 5. Анимация вращения диска со щелью

Для визуального исследования численной диффузии построим анимацию вращения диска с прорезанной щелью.

На изображение численной концентрации накладывается красный контур точного решения, который вращается с угловой скоростью $\omega=2\pi$.

Анимация охватывает пять полных оборотов.

Сравнение точного контура с численным распределением позволяет определить, какие геометрические особенности размываются быстрее всего и как накапливается ошибка со временем.


In [ ]:

# Моменты кадров: от 0 до 5 оборотов
SLOTTED_ANIM_TIMES = np.linspace(0, 5, 51)

print("Подготовка кадров анимации...")

slotted_anim_result = simulate(
    N=200,
    profile="slotted",
    t_end=5.0,
    cfl=0.8,
    observation_times=SLOTTED_ANIM_TIMES,
    snapshot_times=SLOTTED_ANIM_TIMES
)

# Численная сетка и более подробная сетка контура
X, Y, _, _, _, _ = create_grid(200)
Xc, Yc, _, _, _, _ = create_grid(400)

frames = np.round(SLOTTED_ANIM_TIMES, 12)
snapshots = slotted_anim_result["snapshots"]

fig, ax = plt.subplots(figsize=(7, 6), dpi=80)

im = ax.imshow(
    snapshots[frames[0]],
    origin="lower",
    extent=[XMIN, XMAX, YMIN, YMAX],
    cmap="viridis",
    vmin=0,
    vmax=1,
    interpolation="nearest"
)

# Начальный контур точного решения
contour_artist = ax.contour(
    Xc,
    Yc,
    exact_solution(Xc, Yc, frames[0], "slotted"),
    levels=[0.5],
    colors="red",
    linewidths=1.5
)

ax.set_xlim(-0.7, 0.7)
ax.set_ylim(-0.7, 0.7)
ax.set_aspect("equal")

ax.set_xlabel("x")
ax.set_ylabel("y")

fig.colorbar(
    im,
    ax=ax,
    shrink=0.8,
    label="Концентрация"
)


def update_slotted(frame):
    global contour_artist

    t = float(frames[frame])

    # Обновляем численное распределение
    im.set_data(snapshots[frames[frame]])

    # До Matplotlib 3.8 удаляются отдельные коллекции контура.
    if hasattr(contour_artist, "remove"):
        contour_artist.remove()
    else:
        for collection in contour_artist.collections:
            collection.remove()

    U_exact = exact_solution(
        Xc, Yc, t, "slotted"
    )

    contour_artist = ax.contour(
        Xc,
        Yc,
        U_exact,
        levels=[0.5],
        colors="red",
        linewidths=1.5
    )

    ax.set_title(
        f"Диск со щелью, t = {t:.2f}, "
        f"оборотов = {t:.1f}"
    )

    return (im, contour_artist)


anim_slotted = FuncAnimation(
    fig,
    update_slotted,
    frames=len(frames),
    interval=120,
    blit=False,
    repeat=True
)

plt.close(fig)

plt.rcParams["animation.embed_limit"] = 60

display(
    HTML(anim_slotted.to_jshtml(fps=10))
)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, t in zip(axes, [1.0, 5.0]):
    for N in GRID_SIZES:
        X, Y, _, _, _, _ = create_grid(N)

        U = yellow_results["slotted"][N][
            "snapshots"
        ][round(t, 12)]

        # Горизонтальное сечение рядом с y=0
        j = np.argmin(np.abs(Y[:, 0]))

        ax.plot(
            X[j],
            U[j],
            marker=".",
            markersize=3,
            linewidth=1,
            label=f"{N}x{N}"
        )

    # Точное сечение непосредственно на y=0
    x_exact = np.linspace(0.15, 0.65, 1000)
    y_exact = np.zeros_like(x_exact)

    u_exact = exact_solution(
        x_exact, y_exact, t, "slotted"
    )

    ax.plot(
        x_exact,
        u_exact,
        "k--",
        linewidth=2,
        label="Точное решение"
    )

    ax.set_xlim(0.15, 0.65)
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(f"Сечение y=0, t={t}")
    ax.set_xlabel("x")
    ax.set_ylabel("Концентрация")
    ax.grid(alpha=0.3)
    ax.legend()

fig.suptitle(
    "Заполнение щели вследствие численной диффузии",
    fontsize=14
)

plt.tight_layout()
plt.show()



### Ошибка и максимальная концентрация

После одного оборота интегральная ошибка $E_1$ гладкого пятна уменьшается от $0.04896$ на сетке $50\times50$ до $0.03247$ на сетке $200\times200$.

При этом максимальная концентрация увеличивается от $0.0852$ до $0.2804$, приближаясь к точному максимуму, равному $1$.

После пяти оборотов численная диффузия становится значительно сильнее. Для гладкого пятна на сетке $200\times200$ максимальная концентрация составляет всего $0.0692$, а для диска со щелью — $0.2069$.

При этом после пяти оборотов интегральная ошибка не уменьшается монотонно при измельчении сетки. Это связано с совместным влиянием численного размытия, изменения формы и потери примеси через границу. Поэтому для оценки качества решения необходимо использовать не только $E_1$, но и максимум концентрации, а также количество примеси.

### Сохранение количества примеси

Точное решение сохраняет количество примеси, поскольку пятно вращается внутри области, не достигая её границы.

Численная схема является консервативной, однако из-за численной диффузии часть примеси достигает внешней границы и покидает область.

После пяти оборотов потери примеси для гладкого пятна составляют:

- $51.85\%$ на сетке $50\times50$;
- $25.14\%$ на сетке $100\times100$;
- $7.45\%$ на сетке $200\times200$.

Для диска со щелью соответствующие потери равны $52.43\%$, $26.62\%$ и $9.33\%$.

Проверка баланса показала, что

$$
M(t)+Q_{\mathrm{out}}(t)=M(0)
$$

выполняется с абсолютной погрешностью порядка $10^{-17}$.

Следовательно, уменьшение количества примеси внутри расчётной области согласуется с накопленным выходящим граничным потоком. Численного нарушения дискретного закона сохранения массы не наблюдается.
